![](https://github.com/datagong/data/blob/main/datagong.png?raw=true)

© DATAGONG - Tous droits réservés - 2026

📋 **Rappels Conditions Générales d'Utilisation**

⚠️ Les Notebooks sont privés

❌ partage des Notebooks, de leur contenu, des liens, des images, …

❌ publier les Notebooks sur GitHub (public) ou tout autre outil de versionning

✅ sauvegarder les Notebooks dans votre environnement personnel et privé

✅ réutiliser les codes dans le cadre de vos projets en entreprise / projets personnels / etc.

✅ annoter / modifier les Notebooks dans votre environnement personnel et privé

# <center><u><b>Data Visualization avec Streamlit et Plotly</b></u></center>

# <b>Streamlit + Plotly — 2. Ingestion de données & Caching</b>

Dans le notebook précédent, vous avez posé les fondations de votre application : l'arborescence du projet, un premier `app.py` fonctionnel avec un graphique Plotly, et un thème personnalisé. Votre app affichait un jeu de données intégré à Plotly (Iris) — pratique pour démarrer, mais dans la vraie vie, vous travaillerez avec vos propres fichiers.

C'est exactement ce que nous allons faire ici. Nous allons apprendre à **charger des données depuis un fichier CSV**, à organiser cette logique dans un module Python dédié (`utils/data.py`), et surtout à découvrir le **caching** — un mécanisme central de Streamlit qui vous évitera de recharger vos données à chaque interaction utilisateur.

# 0. Générer un petit jeu de données d'exemple

Avant de charger quoi que ce soit, il nous faut des données à charger. Pour que vous puissiez suivre ce notebook sans dépendre d'un fichier externe, nous allons générer nous-mêmes un CSV simple : 120 jours de ventes, réparties entre deux catégories de produits ("A" et "B").

Ce fichier sera enregistré dans `streamlit_app/data/ventes.csv`, là où notre application ira le chercher par la suite.

In [3]:
import pandas as pd
import numpy as np

# Génère une série de dates quotidiennes sur 120 jours
dates = pd.date_range("2023-01-01", periods=120, freq="D")

# Initialise le générateur de nombres aléatoires pour la reproductibilité
rng = np.random.default_rng(42)

# Attribue aléatoirement une catégorie "A" ou "B" à chaque date
cat = np.where(rng.random(len(dates)) > 0.5, "A", "B")

# Génère des ventes aléatoires, avec un bonus pour la catégorie "A"
sales = rng.poisson(lam=100, size=len(dates)) + (cat == "A") * 20

# Crée le DataFrame avec les colonnes date, catégorie et ventes
df = pd.DataFrame({"date": dates, "categorie": cat, "ventes": sales})

# Sauvegarde le DataFrame au format CSV pour l'application Streamlit
df.to_csv("../streamlit_app/data/ventes.csv", index=False)

# Affiche les premières lignes du DataFrame
df.head()


,date,categorie,ventes
0,2023-01-01,A,122
1,2023-01-02,B,83
2,2023-01-03,A,115
3,2023-01-04,A,105
4,2023-01-05,B,102


# 1. Créer le module `utils/data.py`

Jusqu'ici, tout notre code vivait dans `app.py`. C'est suffisant pour un prototype, mais dès que votre application grandit, vous allez vouloir **séparer la logique de chargement et de filtrage des données** dans un fichier à part. C'est le rôle du module `utils/data.py` : un fichier Python classique qui contient des fonctions réutilisables, que vous pourrez importer depuis n'importe quelle page de votre app.

## 1.1. Comprendre le caching

Vous le savez si vous avez travaillé avec des notebooks Jupyter : quand vous exécutez une cellule qui charge un gros fichier, ça peut prendre du temps. Dans un notebook, ce n'est pas très grave — vous exécutez la cellule une fois, les données restent en mémoire, et vous travaillez dessus tranquillement.

Avec Streamlit, c'est différent. À chaque interaction de l'utilisateur (un clic sur un bouton, un changement de filtre…), Streamlit **ré-exécute l'intégralité de votre script** `app.py` de haut en bas. Sans précaution, cela signifie que votre fichier CSV serait rechargé à chaque clic — ce qui est évidemment trop lent quand les données sont volumineuses.

C'est là qu'intervient le **caching** (mise en cache). Le principe est simple : vous demandez à Streamlit de mémoriser le résultat d'une fonction. La première fois qu'elle est appelée, Streamlit l'exécute normalement et stocke le résultat. Les fois suivantes, si les arguments n'ont pas changé, Streamlit renvoie directement le résultat stocké sans ré-exécuter la fonction. En résumé, le caching transforme une opération coûteuse en un accès quasi-instantané.

## 1.2. Les décorateurs de cache Streamlit

Pour activer le caching, Streamlit utilise des **décorateurs**. Si vous n'êtes pas familier avec ce concept, un décorateur est simplement une annotation que vous placez au-dessus d'une fonction avec le symbole `@`. Il modifie le comportement de la fonction sans que vous ayez à toucher à son code.

Streamlit propose deux décorateurs de cache :

`@st.cache_data` est celui que vous utiliserez le plus souvent. Il est conçu pour les fonctions qui retournent des **données sérialisables** — typiquement des DataFrames, des listes, des dictionnaires. À chaque appel, Streamlit vérifie si les arguments sont les mêmes que la dernière fois ; si oui, il renvoie une copie du résultat en cache. Dans notre cas, nous allons l'utiliser pour charger notre CSV une seule fois, même si l'utilisateur interagit des dizaines de fois avec l'interface.

`@st.cache_resource` fonctionne sur le même principe, mais il est pensé pour des **objets non sérialisables** qu'il faut créer une seule fois et partager entre les utilisateurs : des connexions à une base de données, des modèles de machine learning, des clients d'API. Nous n'en aurons pas besoin dans ce notebook, mais gardez-le en tête pour la suite.

⚠️ Attention : le cache est lié aux arguments de la fonction. Si vous changez le chemin du fichier passé en paramètre, Streamlit considérera que c'est un nouvel appel et ré-exécutera la fonction.

📖 Pour aller plus loin : [documentation officielle du caching Streamlit](https://docs.streamlit.io/develop/concepts/architecture/caching)

## 1.3. Le code du module

Le module ci-dessous contient deux fonctions. `load_data` charge le CSV et s'occupe de convertir les types (dates, catégories). `filter_data` applique des filtres sur le DataFrame — nous l'utiliserons dans les prochains notebooks quand nous ajouterons des widgets interactifs.

In [4]:
%%writefile ../streamlit_app/utils/data.py
from __future__ import annotations
from typing import Optional, Iterable
import pandas as pd
import streamlit as st

@st.cache_data(show_spinner=False)
def load_data(path: str = "streamlit_app/data/ventes.csv") -> pd.DataFrame:
    '''Charge le CSV et assure les bons types.'''
    # Lecture du fichier CSV avec conversion de la colonne "date"
    df = pd.read_csv(path, parse_dates=["date"])
    # Conversion de la colonne "categorie" en type category pour optimiser la mémoire
    df["categorie"] = df["categorie"].astype("category")
    return df

def filter_data(df: pd.DataFrame, categorie: Optional[Iterable[str]] = None, date_min=None, date_max=None) -> pd.DataFrame:
    '''Applique des filtres simples et retourne une copie filtrée.'''
    q = "True"
    # Filtre sur les catégories si précisé
    if categorie:
        cats = ",".join([f"'{c}'" for c in categorie])
        q += f" and categorie in [{cats}]"
    # Filtre sur la date minimale si précisé
    if date_min is not None:
        q += f" and date >= @date_min"
    # Filtre sur la date maximale si précisé
    if date_max is not None:
        q += f" and date <= @date_max"
    # Application du filtre avec .query()
    return df.query(q)


Writing ../streamlit_app/utils/data.py


## 2. Utiliser ces fonctions dans `app.py`

Notre module `utils/data.py` est prêt, il ne reste plus qu'à l'intégrer dans l'application. Nous allons ajouter du code à `app.py` pour charger les données, les afficher sous forme de tableau, puis tracer un premier graphique temporel.

Avant de passer au code, prenons quelques instants pour comprendre les méthodes Streamlit que nous allons utiliser.

### `st.spinner`

Quand une opération prend un peu de temps (chargement de fichier, calcul…), il est important de signaler à l'utilisateur que quelque chose se passe. `st.spinner` affiche un petit message animé le temps que le bloc de code s'exécute. Vous l'utilisez comme un context manager Python classique :

```python
with st.spinner("Chargement en cours…"):
    data = load_data()
```

Le message disparaît automatiquement dès que le code à l'intérieur du `with` est terminé. Avec le caching en place, ce spinner ne sera visible que lors du tout premier chargement.

### `st.dataframe`

Pour afficher un DataFrame dans votre app, `st.dataframe` est la méthode la plus courante. Elle génère un tableau interactif que vos utilisateurs peuvent trier et parcourir directement dans le navigateur. Le paramètre `use_container_width=True` permet au tableau de s'adapter à la largeur de la page.

### `st.plotly_chart`

Vous avez déjà rencontré cette méthode dans le notebook précédent : elle prend un objet figure Plotly et l'affiche dans l'app. Ici, nous allons l'utiliser avec un graphique en courbes (`px.line`) pour visualiser l'évolution des ventes par catégorie.

📖 Retrouvez la documentation complète de ces méthodes : [`st.spinner`](https://docs.streamlit.io/develop/api-reference/status/st.spinner) · [`st.dataframe`](https://docs.streamlit.io/develop/api-reference/data/st.dataframe) · [`st.plotly_chart`](https://docs.streamlit.io/develop/api-reference/charts/st.plotly_chart)

**💡 Astuce :** Vous remarquerez le flag `-a` dans `%%writefile -a` : il signifie *append*, c'est-à-dire que le contenu sera **ajouté à la fin** du fichier `app.py` existant, sans écraser ce qui s'y trouve déjà.

In [5]:
%%writefile -a ../streamlit_app/app.py
# --- Section: Données ---
import pandas as pd
from utils.data import load_data, filter_data

# Affiche un spinner pendant le chargement des données
with st.spinner("Chargement des données…"):
    data = load_data()  # Chargement des données avec cache

# Affiche un aperçu interactif des premières lignes du DataFrame
st.write("Aperçu des données :")
st.dataframe(data.head(), use_container_width=True)

# Crée un graphique de l'évolution des ventes par catégorie
fig_line = px.line(data, x="date", y="ventes", color="categorie",
                   title="Ventes journalières")
# Affiche le graphique dans l'application Streamlit
st.plotly_chart(fig_line, use_container_width=True)


Appending to ../streamlit_app/app.py


💡 Pensez à relancer votre app (`streamlit run streamlit_app/app.py`) pour voir le tableau de données et le graphique en courbes apparaître sous votre premier scatter plot.

# <font color='#ff7373'><b>Félicitations !</b></font>

Vous venez de franchir une étape importante : votre application ne dépend plus d'un jeu de données intégré à Plotly, elle charge ses propres données depuis un fichier CSV. Vous avez également mis en place le caching avec `@st.cache_data`, ce qui garantit que vos données ne sont lues qu'une seule fois, quelle que soit l'activité de l'utilisateur sur l'interface.


Dans le prochain notebook, nous allons nous occuper du **design de nos graphiques Plotly** pour obtenir un rendu visuel cohérent et professionnel sur l'ensemble de l'application.

# <center><font color='#3b4859'><u>![](https://github.com/datagong/data/blob/main/mini%20datagong%202.png?raw=true)</u></font></center>